In [0]:
%sql
CREATE CATALOG IF NOT EXISTS streaming_dev
 COMMENT 'Catálogo do projeto Streaming';

CREATE SCHEMA IF NOT EXISTS streaming_dev.bronze_ingestao
 COMMENT 'Dados brutos do projeto';

 CREATE SCHEMA IF NOT EXISTS streaming_dev.silver
 COMMENT 'Dados limpos e tratados';

 CREATE SCHEMA IF NOT EXISTS streaming_dev.gold
 COMMENT 'Agregados para Bi';

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS streaming_dev.bronze_ingestao.raw_files
 COMMENT 'CSVs brutos do projeto';

In [0]:
PATH_BASE = '/Volumes/streaming_dev/bronze_ingestao/raw_files'

dbutils.fs.ls(PATH_BASE)

df = spark.read.option("header", "true").csv(f"{PATH_BASE}/*.csv")
display(df)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

PATH_BASE = '/Volumes/streaming_dev/bronze_ingestao/raw_files'

def ingest_bronze(nome: str, destino: str):
    df = (spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("multiLine", "true")
        .option("escape", '"')
        .csv(f"{PATH_BASE}/{nome}"))

    df = (df
      .withColumn("ingest_timestamp", current_timestamp())
      .withColumn("source_file", lit(nome)))

    df.write.mode("overwrite").saveAsTable(destino)
    print(f" {destino}: {df.count()} linhas")

ingest_bronze("catalogo_filmes.csv", "streaming_dev.bronze_ingestao.catalogo_filmes")
ingest_bronze("logs_streaming.csv",  "streaming_dev.bronze_ingestao.logs_streaming")
ingest_bronze("usuarios.csv",        "streaming_dev.bronze_ingestao.usuarios")


In [0]:
%sql
select * from streaming_dev.bronze_ingestao.catalogo_filmes limit 20;